# JanoGPT Training on Kaggle

This notebook trains a GPT-2 model using JanoGPT on Kaggle's free GPU.

**What this notebook does:**
1. Downloads JanoGPT code from GitHub
2. Installs dependencies
3. Sets up WandB for tracking
4. Downloads OpenWebText dataset
5. Trains GPT-2 124M for 1000 steps
6. Saves checkpoint

**Requirements:**
- Enable GPU in Kaggle (Settings → Accelerator → GPU T4 x2)
- Add WandB API key as Kaggle secret (optional)

**Training time:** ~30-60 minutes on T4 GPU

## 1. Setup Environment

In [ ]:
# Check GPU availability
# !nvidia-smi

In [ ]:
# Clone JanoGPT repository
!git clone https://github.com/hhe0u0/janogpt
%cd janogpt
!pip install -e .

In [ ]:
# Install dependencies
# !pip install -q jax[cuda12]

In [ ]:
# Verify JAX sees the GPU
import jax
print(f"JAX devices: {jax.devices()}")
print(f"Device count: {jax.local_device_count()}")
print(f"Device type: {jax.devices()[0].platform}")

## 2. Setup WandB (Optional)

WandB tracks your training metrics. You can:
- **Option A:** Use Kaggle Secrets (recommended)
  1. Go to Kaggle Settings → Add-ons → Secrets
  2. Add secret: Name=`WANDB_API_KEY`, Value=`your_wandb_key`
  3. Enable the secret for this notebook
- **Option B:** Login interactively (paste your key when prompted)
- **Option C:** Skip WandB (set `wandb_enabled = False` below)

In [ ]:
import os

# Configuration
wandb_enabled = True  # Set to False to disable WandB

if wandb_enabled:
    try:
        # Try to get WandB key from Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        wandb_key = user_secrets.get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = wandb_key
        print("✓ Loaded WandB API key from Kaggle secrets")
    except:
        print("⚠️  WandB secret not found. Trying interactive login...")
        import wandb
        wandb.login()
else:
    print("ℹ️  WandB tracking disabled")

## 3. Setup Data Paths

We'll use the OpenWebText dataset that's already available in Kaggle.

**Dataset location:** `/kaggle/input/datasets/windmaple/openwebtext-gpt2/`

In [ ]:
# Check if OpenWebText dataset is available
from pathlib import Path
import os

# Kaggle dataset path
DATA_DIR = "/kaggle/input/datasets/windmaple/openwebtext-gpt2"
data_dir = Path(DATA_DIR)

if data_dir.exists():
    train_bin = data_dir / "train.bin"
    val_bin = data_dir / "val.bin"
    
    if train_bin.exists() and val_bin.exists():
        print(f"✓ Found OpenWebText dataset at {DATA_DIR}")
        print(f"  train.bin: {train_bin.stat().st_size / 1e9:.2f} GB")
        print(f"  val.bin: {val_bin.stat().st_size / 1e6:.2f} MB")
    else:
        print(f"⚠️  Dataset path exists but files not found")
        print(f"  Expected: {train_bin}")
        print(f"  Expected: {val_bin}")
else:
    print(f"⚠️  Dataset not found at {DATA_DIR}")
    print("\nTo add the dataset to your Kaggle notebook:")
    print("1. Click 'Add data' in the right sidebar")
    print("2. Search for 'openwebtext-gpt2'")
    print("3. Add 'windmaple/openwebtext-gpt2'")
    print("4. The dataset will be available at /kaggle/input/datasets/windmaple/openwebtext-gpt2/")

In [ ]:
# Create symbolic links to data
from pathlib import Path
import os

DATA_DIR = "/kaggle/input/datasets/windmaple/openwebtext-gpt2"
data_dir = Path(DATA_DIR)

# Create work directory
work_data_dir = Path("data/openwebtext")
work_data_dir.mkdir(parents=True, exist_ok=True)

if data_dir.exists():
    train_bin = data_dir / "train.bin"
    val_bin = data_dir / "val.bin"
    
    # Create symbolic links (fast, no copying)
    if not (work_data_dir / "train.bin").exists():
        os.symlink(train_bin, work_data_dir / "train.bin")
    if not (work_data_dir / "val.bin").exists():
        os.symlink(val_bin, work_data_dir / "val.bin")
    
    print(f"✓ Data linked to {work_data_dir}")
    print(f"  train.bin -> {train_bin}")
    print(f"  val.bin -> {val_bin}")
else:
    print(f"⚠️  Creating dummy data for testing...")
    import numpy as np
    
    # 10MB of random tokens (for testing only)
    dummy_train = np.random.randint(0, 50257, size=5_000_000, dtype=np.uint16)
    dummy_val = np.random.randint(0, 50257, size=500_000, dtype=np.uint16)
    
    dummy_train.tofile(work_data_dir / "train.bin")
    dummy_val.tofile(work_data_dir / "val.bin")
    
    print(f"✓ Created dummy dataset at {work_data_dir}")
    print("  (This is for testing only - add the real dataset for training)")

## 4. Create Training Configuration

**Dynamic Configuration:**
- Auto-detects number of GPUs
- `micro_batch_size: 1` - Safe for 16GB T4 GPU
- `gradient_accumulation_steps` - Calculated dynamically to target ~0.5M tokens
- Formula: `micro × accum × devices × seq_len = 500K tokens`

**For 2x T4 GPUs:**
- Effective batch: 1 × 244 × 2 × 1024 = **499,712 tokens ≈ 0.5M** ✓

**For 1x T4 GPU:**
- Effective batch: 1 × 488 × 1 × 1024 = **499,712 tokens ≈ 0.5M** ✓

In [ ]:
import json
from pathlib import Path
import jax

# Dynamically detect number of GPUs
num_devices = jax.local_device_count()
print(f"Detected {num_devices} GPU(s)")

# Conservative batch size for GPU (T4 16GB)
micro_batch_size = 1  # Safe for 16GB GPU

# Calculate gradient accumulation to target ~0.5M tokens per step
# Formula: micro_batch × accum_steps × num_devices × seq_len = target_tokens
target_tokens = 500_000
seq_len = 1024
gradient_accumulation_steps = target_tokens // (micro_batch_size * num_devices * seq_len)

effective_tokens = micro_batch_size * gradient_accumulation_steps * num_devices * seq_len

print(f"\nBatch configuration:")
print(f"  micro_batch_size: {micro_batch_size} (per GPU)")
print(f"  gradient_accumulation: {gradient_accumulation_steps}")
print(f"  num_devices: {num_devices}")
print(f"  Effective batch: {effective_tokens:,} tokens per step (~{effective_tokens/1e6:.2f}M)")

# Training configuration for Kaggle GPU
config = {
    "_comment": f"GPT-2 124M training config for Kaggle ({num_devices} GPU(s), 1000 steps)",
    
    "model": {
        "dropout_prob": 0.1,
        "num_blocks": 12,
        "emb_dim": 768,
        "num_heads": 12,
        "seq_len": seq_len,
        "epsilon": 1e-6,
        "voc_size": 50304
    },
    
    "optimizer": {
        "learning_rate": 6e-4,
        "min_learning_rate": 6e-5,
        "warmup_steps": 100,
        "beta1": 0.9,
        "beta2": 0.95,
        "grad_clip": 1.0,
        "weight_decay": 0.1
    },
    
    "training": {
        "max_steps": 1000,
        "micro_batch_size": micro_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "num_devices": num_devices,
        "seed": 42
    },
    
    "data": {
        "data_dir": "data/openwebtext",
        "train_file": "train.bin",
        "val_file": "val.bin"
    },
    
    "logging": {
        "eval_interval": 100,
        "eval_iters": 50,
        "log_interval": 10
    },
    
    "checkpointing": {
        "save_interval": 250,
        "output_dir": "output_kaggle_1k",
        "resume_from_checkpoint": None
    },
    
    "wandb": {
        "enabled": wandb_enabled,
        "project": "janogpt-kaggle",
        "run_name": f"gpt2-124m-{num_devices}gpu-1k",
        "tags": ["kaggle", "gpt2", "1000-steps", f"{num_devices}xGPU"]
    }
}

# Save config
config_dir = Path("configs")
config_dir.mkdir(exist_ok=True)
config_path = config_dir / "train_kaggle_1k.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"\n✓ Config saved to {config_path}")
print("\nConfiguration:")
print(f"  Model: GPT-2 124M ({config['model']['num_blocks']} layers, {config['model']['emb_dim']} dim)")
print(f"  GPUs: {num_devices}")
print(f"  Training: {config['training']['max_steps']} steps")
print(f"  Effective batch: {effective_tokens / 1000:.0f}K tokens per step")
print(f"  Learning rate: {config['optimizer']['learning_rate']}")
print(f"  Checkpoints: Every {config['checkpointing']['save_interval']} steps")
print(f"  WandB: {'Enabled' if config['wandb']['enabled'] else 'Disabled'}")

## 5. Train the Model

Now we'll train for 1000 steps. This should take about 30-60 minutes on a T4 GPU.

**Expected performance:**
- Tokens/sec: ~1000-2000
- Steps/sec: ~0.5-1.0
- Memory usage: ~10-15GB GPU

**Checkpoints will be saved to:** `output_kaggle_1k/checkpoints/`

In [ ]:
# Start training
!python scripts/train.py --config configs/train_kaggle_1k.json

## 6. Verify Checkpoint

Let's verify that the checkpoint was saved correctly.

In [ ]:
# List checkpoints
import os
from pathlib import Path

checkpoint_dir = Path("output_kaggle_1k/checkpoints")

if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("step_*"))
    print(f"✓ Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        # Check size
        total_size = sum(f.stat().st_size for f in ckpt.rglob("*") if f.is_file())
        size_mb = total_size / 1e6
        print(f"  {ckpt.name}: {size_mb:.1f} MB")
    
    # Show latest checkpoint
    if checkpoints:
        latest = checkpoints[-1]
        print(f"\n✓ Latest checkpoint: {latest}")
        print(f"  Files:")
        for f in sorted(latest.rglob("*")):
            if f.is_file():
                print(f"    {f.relative_to(latest)}")
else:
    print("⚠️  No checkpoints found")

## 7. Test the Checkpoint

Let's test text generation with the trained checkpoint.

In [ ]:
# Find the latest checkpoint
checkpoint_dir = Path("output_kaggle_1k/checkpoints")
checkpoints = sorted(checkpoint_dir.glob("step_*"))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Testing with checkpoint: {latest_checkpoint}")
    
    # Generate text
    !python scripts/generate.py \
        --checkpoint {latest_checkpoint} \
        --prompt "Once upon a time" \
        --max_tokens 100 \
        --temperature 0.8
else:
    print("No checkpoint found to test")

## 8. Download Checkpoint (Optional)

You can download the checkpoint to use it later or on another system.

In [ ]:
# Create a zip file of the latest checkpoint
import shutil

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    zip_path = f"{latest_checkpoint.name}.zip"
    
    print(f"Creating archive: {zip_path}")
    shutil.make_archive(
        latest_checkpoint.name,
        'zip',
        checkpoint_dir,
        latest_checkpoint.name
    )
    
    zip_size = Path(zip_path).stat().st_size / 1e6
    print(f"✓ Archive created: {zip_path} ({zip_size:.1f} MB)")
    print(f"\nTo download: Click on the file in the Kaggle output panel")
else:
    print("No checkpoint to archive")

## 9. Upload Checkpoints to Google Drive

Mount your Google Drive and upload checkpoints so they persist beyond the Kaggle session.

**Note:** This requires the `google-colab` package which may not be available in Kaggle. If it fails, skip this step and use the zip download method above.

In [ ]:
# Install google-colab if not available (may fail in Kaggle)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_available = True
    print("✓ Google Drive mounted")
except:
    print("⚠️  Google Drive not available in Kaggle")
    print("   Use the zip download method above instead")
    drive_available = False

if drive_available:
    import shutil
    from pathlib import Path
    import time
    
    # Create Drive directory
    drive_checkpoint_dir = Path("/content/drive/MyDrive/janogpt_checkpoints/kaggle_1k")
    drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    # Upload checkpoints
    local_checkpoint_dir = Path("output_kaggle_1k/checkpoints")
    
    if local_checkpoint_dir.exists():
        checkpoints = sorted(local_checkpoint_dir.glob("step_*"))
        print(f"\nUploading {len(checkpoints)} checkpoint(s) to Drive...")
        
        for ckpt in checkpoints:
            dest = drive_checkpoint_dir / ckpt.name
            
            if dest.exists():
                print(f"  {ckpt.name}: Already in Drive")
            else:
                print(f"  {ckpt.name}: Uploading...", end="", flush=True)
                start = time.time()
                shutil.copytree(ckpt, dest)
                elapsed = time.time() - start
                size_mb = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file()) / 1e6
                print(f" ✓ ({size_mb:.1f} MB, {elapsed:.1f}s)")
        
        print(f"\n✓ All checkpoints uploaded to: {drive_checkpoint_dir}")
    else:
        print("No checkpoints to upload")